We test the functions from our metrics file on 10 made-up houses where the right answers can be counted
by hand. The expected results are written next to each check; the table at the end
shows that `evaluate` returns one row for houses and one for all buildings.

In [1]:
from metrics import share_above_in_top, share_captured, mae, evaluate

# 10 made-up houses. Four are truly above 450: positions 0, 1, 2 and 5.
y_true = [600, 500, 470, 300, 200, 480, 350, 250, 400, 100]
score  = [590, 520, 300, 310, 210, 400, 360, 260, 410, 110]

# The top 20% = 2 houses with the highest score: 590 and 520 (positions 0 and 1).
# Both are truly above 450, so the selection measure should be 2/2 = 1.0
print(share_above_in_top(y_true, score, top=0.20))   # expect 1.0

# They capture 2 of the 4 houses above 450: 2/4 = 0.5
print(share_captured(y_true, score, top=0.20))       # expect 0.5

# The top 40% = 4 houses: positions 0, 1, 8 (410), 5 (400).
# Truly above 450: positions 0, 1 and 5, so 3/4 = 0.75
print(share_above_in_top(y_true, score, top=0.40))   # expect 0.75

# MAE: a perfect prediction must give 0
print(mae(y_true, y_true))                           # expect 0.0

# The full table: half "houses", half not
is_house = [True] * 5 + [False] * 5
evaluate(y_true, score, is_house)

1.0
0.5
0.75
0.0


,group,n,share_above_top10,captured_top10,share_above_top20,captured_top20,mae
0,houses,5,1.0,0.333333,1.0,0.333333,44.0
1,all buildings,10,1.0,0.250000,1.0,0.500000,34.0


## Baselines

Two simple approaches that every model must beat, scored on the same 5 folds and with the same measures as the models (task 20):
- **Training mean:** every building gets the average index of the training folds. All scores are equal, so the ranking is random; this is a sanity check (houses: about 38% above 450 in the top 20%, about 20% captured).
- **Expert rule:** every building gets the average index of its construction period × energy group in the training folds (feedback point 4: can the models beat it?). A combination not seen in training gets the overall mean.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from cv import summarize
from pipeline import ExpertRuleEncoder, RULE_COLUMNS

processed = Path("../data/processed")
tables = Path("../output/tables")
tables.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(processed / "building_clean.csv")
split = pd.read_csv(processed / "split.csv")
data = data.merge(split, on="EGID", how="inner", validate="one_to_one")

train = data[data["test"] == False].reset_index(drop=True)   # test set stays closed until task 26
train["fold"] = train["fold"].astype(int)

y = train["y"].to_numpy(dtype=float)
is_house = train["house"].to_numpy(dtype=bool)
folds = train["fold"].to_numpy()

In [3]:
pred_mean = np.full(len(train), np.nan)   # one prediction per building, filled fold by fold
pred_rule = np.full(len(train), np.nan)
per_fold = {"Training mean": [], "Expert rule": []}

for k in range(5):
    tr, va = folds != k, folds == k   # training = the other 4 folds, validation = fold k

    pred_mean[va] = y[tr].mean()      # training mean
    rule = ExpertRuleEncoder(smoothing=0).fit(train.loc[tr, RULE_COLUMNS], y[tr])   # table of averages from training only
    pred_rule[va] = rule.transform(train.loc[va, RULE_COLUMNS]).ravel()             # look up each validation building

    for name, pred in [("Training mean", pred_mean), ("Expert rule", pred_rule)]:
        result = evaluate(y[va], pred[va], is_house[va])
        result.insert(0, "fold", k)
        per_fold[name].append(result)

In [4]:
rows = []
for name, parts in per_fold.items():
    summary = summarize(pd.concat(parts, ignore_index=True))   # mean and SE over the 5 folds, as in task 20
    for _, s in summary.iterrows():
        rows.append({
            "model": name,
            "group": s["group"],
            "top 20%: above 450 (%)": round(s["share_above_top20_mean"] * 100, 1),
            "± SE": round(s["share_above_top20_se"] * 100, 1),
            "top 10%: above 450 (%)": round(s["share_above_top10_mean"] * 100, 1),
            "top 20%: captured (%)": round(s["captured_top20_mean"] * 100, 1),
            "MAE": round(s["mae_mean"], 1),
        })
baselines = pd.DataFrame(rows)
baselines

,model,group,top 20%: above 450 (%),± SE,top 10%: above 450 (%),top 20%: captured (%),MAE
0,Training mean,houses,34.0,3.9,34.5,18.0,117.5
1,Training mean,all buildings,28.8,0.9,27.1,19.9,94.8
2,Expert rule,houses,45.1,3.5,43.9,23.8,106.1
3,Expert rule,all buildings,47.1,1.3,51.7,32.5,80.4


In [ ]:
baselines.to_csv(tables / "task18_baselines.csv", index=False)
oof = train[["EGID", "fold", "house", "y"]].copy()
oof["Training mean"] = pred_mean
oof["Expert rule"] = pred_rule
assert oof[["Training mean", "Expert rule"]].notna().all().all(), "a building has no prediction"
oof.to_csv(tables / "task18_oof_predictions.csv", index=False)
print("Saved task18_baselines.csv and task18_oof_predictions.csv:", len(oof), "buildings")


Saved task18_baselines.csv and task18_oof_predictions.csv: 12710 buildings
